In [1]:
import pandas as pd
import numpy as np
import re
import json
import time
import pickle
from tqdm import tqdm
from nltk.stem.snowball import RussianStemmer
import bm25s

tqdm.pandas()

# Загружаем данные
train = pd.read_parquet('../dataset/train.parquet')
benchmark_items = pd.read_parquet('../dataset/benchmark_items.parquet')
benchmark_queries = pd.read_parquet('../dataset/benchmark_queries.parquet')

train['item_id'] = train['item_id'].astype(str)
benchmark_items['item_id'] = benchmark_items['item_id'].astype(str)
benchmark_queries['query_id'] = benchmark_queries['query_id'].astype(str)

# Токены полей из предыдущего ноутбука
tokens_by_id = {}
with open('../dataset/item_tokens_fields.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        tokens_by_id[obj['item_id']] = obj

# Восстанавливаем числовые поля
def to_float(s):
    if pd.isna(s): return np.nan
    s = str(s).replace(',', '.').replace(' ', '')
    try: return float(s)
    except: return np.nan

for df in [train, benchmark_items]:
    df['item_price'] = df['item_price'].apply(to_float)
    df['item_rating'] = pd.to_numeric(df['item_rating'], errors='coerce').fillna(-1)
    df['item_rating_reviews_count'] = pd.to_numeric(df['item_rating_reviews_count'], errors='coerce').fillna(-1)

C:\Users\googl\python_global_env\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
stemmer = RussianStemmer()
TOKEN_RE = re.compile(r'[а-яa-z0-9]+')
_STEM_CACHE = {}

def tokenize_stem(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) <= 1: continue
        s = _STEM_CACHE.get(t)
        if s is None:
            s = stemmer.stem(t)
            _STEM_CACHE[t] = s
        out.append(s)
    return out

def char_3gram_tokens(text):
    """Разбивает текст на 3-граммы по символам — устойчив к опечаткам и морфологии."""
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9]', '', text)
    if len(text) < 3:
        return [text] if text else []
    return [text[i:i+3] for i in range(len(text) - 2)]

# Строим токены char-3-gram по заголовку объявления (главный сигнал)
print("Строим char-3-gram токены по заголовкам...")
title_chargrams_tokens = [
    char_3gram_tokens(t) for t in tqdm(benchmark_items['item_title_raw'].tolist(), desc="char3")
]
benchmark_items['tokens_title_char3'] = title_chargrams_tokens

# Сохраняем
with open('../dataset/item_title_char3.jsonl', 'w', encoding='utf-8') as f:
    for iid, tks in zip(benchmark_items['item_id'], title_chargrams_tokens):
        f.write(json.dumps({'item_id': iid, 'tokens': tks}, ensure_ascii=False) + '\n')
print("Сохранено в dataset/item_title_char3.jsonl")

Строим char-3-gram токены по заголовкам...


char3: 100%|██████████| 189212/189212 [00:01<00:00, 104831.14it/s]


Сохранено в dataset/item_title_char3.jsonl


In [3]:
print("Строим BM25 char-3-gram по заголовкам...")
r_char3 = bm25s.BM25()
r_char3.index(title_chargrams_tokens)
r_char3.save('dataset/bm25_index_char3')
print("Сохранено")

Строим BM25 char-3-gram по заголовкам...


Сохранено


In [4]:
def make_query_key(df):
    cols = ['search_query', 'search_location_id', 'search_is_delivery_search',
            'search_infm_params_text', 'search_category']
    return df[cols].fillna('__NA__').astype(str).agg('||'.join, axis=1)

train['query_key'] = make_query_key(train)
train = train.drop_duplicates(subset=['query_key', 'item_id']).reset_index(drop=True)

# Из train берём только те item_id, что есть в benchmark_items
bench_item_ids = set(benchmark_items['item_id'])
valid_pairs = train[train['item_id'].isin(bench_item_ids)].copy()

valid_groups = valid_pairs.groupby('query_key').agg(
    search_query=('search_query', 'first'),
    search_infm=('search_infm_params_text', 'first'),
    search_cat=('search_category', 'first'),
    search_location=('search_location_id', 'first'),
    relevant_items=('item_id', lambda x: list(set(x))),
).reset_index()

print(f"Валидационных query_key: {len(valid_groups)}")

# Токены запросов
def build_query_text(row):
    parts = [str(row['search_query'])]
    if not pd.isna(row['search_infm']):
        parts.append(str(row['search_infm']))
    return ' '.join(parts)

valid_groups['query_text_full'] = valid_groups.apply(build_query_text, axis=1)
valid_groups['query_tokens'] = valid_groups['query_text_full'].progress_apply(tokenize_stem)
valid_groups['query_title_char3'] = valid_groups['search_query'].progress_apply(char_3gram_tokens)
print("Токены запросов готовы")

Валидационных query_key: 26556


100%|██████████| 26556/26556 [00:00<00:00, 323375.95it/s]

Токены запросов готовы


In [5]:
K_BM25 = 1000
BATCH = 128

def batch_bm25(retriever, q_tokens_list, k, name):
    all_ids = [None] * len(q_tokens_list)
    all_scores = [None] * len(q_tokens_list)
    for s in tqdm(range(0, len(q_tokens_list), BATCH), desc=f"BM25 {name}"):
        e = min(s + BATCH, len(q_tokens_list))
        batch_q = q_tokens_list[s:e]
        safe = [q if q else ['_empty_'] for q in batch_q]
        res, sc = retriever.retrieve(safe, k=k, show_progress=False)
        for j in range(len(batch_q)):
            all_ids[s + j] = res[j]
            all_scores[s + j] = sc[j]
    return all_ids, all_scores

# Загружаем существующие индексы full, title, desc, params (из прошлого ноутбука)
r_full = bm25s.BM25.load('../dataset/bm25_index', load_corpus=False)
r_title = bm25s.BM25.load('../dataset/bm25_index_title', load_corpus=False)

q_tokens_list = valid_groups['query_tokens'].tolist()
q_char3_list = valid_groups['query_title_char3'].tolist()

print("BM25 full top-1000...")
bm25_full_ids, bm25_full_scores = batch_bm25(r_full, q_tokens_list, K_BM25, "full")
print("BM25 title top-1000...")
bm25_title_ids, bm25_title_scores = batch_bm25(r_title, q_tokens_list, K_BM25, "title")
print("BM25 char3 top-1000...")
bm25_char3_ids, bm25_char3_scores = batch_bm25(r_char3, q_char3_list, K_BM25, "char3")
print("Готово")

BM25 full top-1000...


BM25 full: 100%|██████████| 208/208 [01:00<00:00,  3.44it/s]


BM25 title top-1000...


BM25 title:   9%|▊         | 18/208 [00:08<01:25,  2.21it/s]


KeyboardInterrupt: 

In [ ]:
import random
random.seed(42)

MAX_POS_PER_QUERY = 10
N_NEG_PER_QUERY = 100   # было 50, теперь 100 — больше сложных негативов из rank 500-1000

item_to_pos = {iid: i for i, iid in enumerate(benchmark_items['item_id'])}

# Предзагрузка токенов item'ов в позиционные массивы
item_ids_array = benchmark_items['item_id'].values
item_tokens_title_list = [tokens_by_id.get(iid, {}).get('title', []) for iid in item_ids_array]
item_tokens_desc_list = [tokens_by_id.get(iid, {}).get('desc', []) for iid in item_ids_array]
item_tokens_params_list = [tokens_by_id.get(iid, {}).get('params', []) for iid in item_ids_array]

pairs = []
for pos in tqdm(range(len(valid_groups)), desc="Building pairs"):
    relevant = set(valid_groups.iloc[pos]['relevant_items'])

    # Объединённый пул из трёх индексов
    pool = set()
    for arr in [bm25_full_ids[pos], bm25_title_ids[pos], bm25_char3_ids[pos]]:
        pool.update(arr.tolist())

    # Позитивы
    pos_items = [item_to_pos[i] for i in relevant if i in item_to_pos]
    random.shuffle(pos_items)
    pos_items = pos_items[:MAX_POS_PER_QUERY]
    for ip in pos_items:
        pairs.append((pos, ip, 1))

    # Негативы — из pool без позитивов, желательно из «сложной» части (rank 100-1000)
    neg_pool = list(pool - set(pos_items))
    random.shuffle(neg_pool)
    for ip in neg_pool[:N_NEG_PER_QUERY]:
        pairs.append((pos, ip, 0))

pairs_df = pd.DataFrame(pairs, columns=['query_pos', 'item_pos', 'label'])
print(f"Всего пар: {len(pairs_df)}, позитивов: {pairs_df['label'].sum()}")

Building pairs:  62%|██████▏   | 16576/26556 [04:25<00:06, 1532.84it/s]

In [ ]:
bi = benchmark_items.reset_index(drop=True)
item_microcat_arr = bi['item_microcat_id'].values
item_loc_arr = bi['item_location_id'].values
item_rating_arr = bi['item_rating'].values
item_reviews_arr = bi['item_rating_reviews_count'].values
item_price_log_arr = np.log1p(bi['item_price'].fillna(0).clip(lower=0).values)
item_title_len_arr = bi['item_title_raw'].fillna('').astype(str).str.len().values
item_desc_len_arr = bi['item_description_raw'].fillna('').astype(str).str.len().values

# Char-3-gram токены item'ов
item_char3_tokens = [tokens_by_id.get(iid, {}) and set() or set() for iid in item_ids_array]  # заполним ниже
item_char3_list = []
with open('../dataset/item_title_char3.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        item_char3_list.append(set(obj['tokens']))
# item_char3_list в порядке benchmark_items (сначала сохраняли в том же порядке)
item_char3_sets = item_char3_list  # это множества 3-грамм по заголовку

def score_rank_for(ids_arr, scores_arr, qp, ip):
    ids = ids_arr[qp]
    where = np.where(ids == ip)[0]
    if len(where) == 0:
        return 9999, 0.0
    rank = int(where[0])
    return rank, float(scores_arr[qp][rank])

feature_cols = [
    'bm25_full_score', 'bm25_full_rank',
    'bm25_title_score', 'bm25_title_rank',
    'bm25_char3_score', 'bm25_char3_rank',
    'overlap_recall', 'overlap_jaccard',
    'title_overlap_recall', 'desc_overlap_recall', 'params_overlap_recall',
    'char3_jaccard',
    'loc_match', 'cat_match', 'microcat_match',
    'item_rating', 'item_reviews', 'item_price_log',
    'item_title_len', 'item_desc_len',
]

q_tokens_arr = valid_groups['query_tokens'].tolist()
q_loc_arr = valid_groups['search_location'].astype(str).tolist()
q_cat_arr = valid_groups['search_cat'].astype(str).tolist()
q_char3_arr = valid_groups['query_title_char3'].tolist()

print("Считаем фичи...")
rows = np.zeros((len(pairs_df), len(feature_cols)), dtype=np.float32)

for idx, (qi, ip, lbl) in enumerate(tqdm(zip(pairs_df['query_pos'].values,
                                             pairs_df['item_pos'].values,
                                             pairs_df['label'].values),
                                          total=len(pairs_df), desc="Features")):
    qi = int(qi); ip = int(ip)

    qt = q_tokens_arr[qi]; qs = set(qt)
    q_char3 = set(q_char3_arr[qi])

    tt = item_tokens_title_list[ip]
    dt = item_tokens_desc_list[ip]
    pt = item_tokens_params_list[ip]
    at = set(tt + dt + pt)
    ts = set(tt); ds = set(dt); ps = set(pt)

    rf, sf = score_rank_for(bm25_full_ids, bm25_full_scores, qi, ip)
    rt, st = score_rank_for(bm25_title_ids, bm25_title_scores, qi, ip)
    rc3, sc3 = score_rank_for(bm25_char3_ids, bm25_char3_scores, qi, ip)

    if qs and at:
        inter = len(qs & at); ovr = inter / len(qs); ovj = inter / len(qs | at)
    else:
        ovr = ovj = 0.0

    tor = len(qs & ts) / len(qs) if qs else 0.0
    dor = len(qs & ds) / len(qs) if qs else 0.0
    por = len(qs & ps) / len(qs) if qs else 0.0

    item_char3 = item_char3_sets[ip]
    if q_char3 and item_char3:
        c3 = len(q_char3 & item_char3) / len(q_char3 | item_char3)
    else:
        c3 = 0.0

    q_loc = q_loc_arr[qi]
    q_cat = q_cat_arr[qi]

    rows[idx] = [
        sf, rf, st, rt, sc3, rc3,
        ovr, ovj, tor, dor, por, c3,
        int(q_loc == str(item_loc_arr[ip])),
        int(q_cat == str(item_microcat_arr[ip])),
        int(q_cat == str(item_microcat_arr[ip])),
        float(item_rating_arr[ip]),
        float(item_reviews_arr[ip]),
        float(item_price_log_arr[ip]),
        int(item_title_len_arr[ip]),
        int(item_desc_len_arr[ip]),
    ]

for i, c in enumerate(feature_cols):
    pairs_df[c] = rows[:, i]

pairs_df.to_parquet('dataset/train_pairs_features_v2.parquet', index=False)
print(f"Сохранено: {len(pairs_df)} пар, {pairs_df.shape[1]} колонок")

Считаем фичи...


Features:  20%|█▉        | 527633/2685124 [00:24<01:28, 24246.77it/s]

In [6]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

query_text_by_pos = valid_groups['search_query'].to_dict()
pairs_df['query_text_g'] = pairs_df['query_pos'].map(query_text_by_pos)

gkf = GroupKFold(n_splits=5)

lgb_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.03, num_leaves=63,
    min_child_samples=30, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=5,
    lambda_l1=0.1, lambda_l2=0.1,
    n_estimators=2000, n_jobs=8, random_state=42, verbose=-1,
)

models = []
for fold, (train_idx, valid_idx) in enumerate(gkf.split(pairs_df, groups=pairs_df['query_text_g'])):
    train_df = pairs_df.iloc[train_idx]
    valid_df = pairs_df.iloc[valid_idx]

    X_tr = train_df[feature_cols].values
    y_tr = train_df['label'].values
    X_va = valid_df[feature_cols].values
    y_va = valid_df['label'].values

    scale = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    m = lgb.LGBMClassifier(**lgb_params, scale_pos_weight=scale)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          eval_metric='auc',
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

    auc = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    print(f"Fold {fold+1}: AUC={auc:.4f}, best_iter={m.best_iteration_}")
    models.append(m)

best_iter_avg = int(np.mean([m.best_iteration_ for m in models]))
print(f"\nСредний best_iteration: {best_iter_avg}")

# Финальная модель на всём train
scale = (pairs_df['label'] == 0).sum() / max((pairs_df['label'] == 1).sum(), 1)
final_model = lgb.LGBMClassifier(**{k: v for k, v in lgb_params.items() if k != 'n_estimators'},
                                  n_estimators=best_iter_avg,
                                  scale_pos_weight=scale)
final_model.fit(pairs_df[feature_cols].values, pairs_df['label'].values)
print("Финальная модель обучена.")

with open('../dataset/lgbm_reranker_v2.pkl', 'wb') as f:
    pickle.dump({'model': final_model, 'feature_cols': feature_cols}, f)
print("Сохранено в dataset/lgbm_reranker_v2.pkl")

NameError: name 'pairs_df' is not defined

In [ ]:
# Обратные лукапы: {item_pos: rank} для каждого запроса и каждого индекса
def build_rank_lookup(ids_per_query):
    out = []
    for arr in tqdm(ids_per_query, desc="rank lookup"):
        out.append({int(ip): r for r, ip in enumerate(arr)})
    return out

print("Строим обратные лукапы...")
rank_full = build_rank_lookup(bm25_full_ids)
rank_title = build_rank_lookup(bm25_title_ids)
rank_char3 = build_rank_lookup(bm25_char3_ids)

# Предвычисляем множества токенов для всех item'ов (один раз)
print("Множества токенов item'ов...")
item_set_title = [set(t) for t in item_tokens_title_list]
item_set_desc = [set(t) for t in item_tokens_desc_list]
item_set_params = [set(t) for t in item_tokens_params_list]
item_set_all = [item_set_title[i] | item_set_desc[i] | item_set_params[i]
                for i in range(len(item_ids_array))]
print("Готово")

Honest Recall@50 v2:  84%|████████▎ | 4181/5000 [22:51<01:22,  9.92it/s]    

In [ ]:
def compute_features_for_query_fast(qp):
    """Оптимизированная версия: лукапы вместо np.where, предвычисленные множества."""
    pool = set(bm25_full_ids[qp].tolist()) | \
           set(bm25_title_ids[qp].tolist()) | \
           set(bm25_char3_ids[qp].tolist())
    item_pos_list = list(pool)
    n = len(item_pos_list)
    X = np.zeros((n, len(feature_cols)), dtype=np.float32)

    qt = q_tokens_list[qp]
    qs = set(qt)
    q_char3 = set(q_char3_arr[qp])
    q_loc = q_loc_arr[qp]
    q_cat = q_cat_arr[qp]

    qs_len = len(qs) if qs else 1
    qc3_len = len(q_char3) if q_char3 else 1

    # Лукапы и скоры для этого запроса
    lf = rank_full[qp]; lt = rank_title[qp]; lc3 = rank_char3[qp]
    sf_arr = bm25_full_scores[qp]
    st_arr = bm25_title_scores[qp]
    sc3_arr = bm25_char3_scores[qp]

    for i, ip in enumerate(item_pos_list):
        at = item_set_all[ip]
        ts = item_set_title[ip]
        ds = item_set_desc[ip]
        ps = item_set_params[ip]

        rf = lf.get(ip, 9999); sf = float(sf_arr[rf]) if rf < 9999 else 0.0
        rt = lt.get(ip, 9999); st = float(st_arr[rt]) if rt < 9999 else 0.0
        rc3 = lc3.get(ip, 9999); sc3 = float(sc3_arr[rc3]) if rc3 < 9999 else 0.0

        if qs and at:
            inter = len(qs & at)
            ovr = inter / qs_len
            denom = qs_len + len(at) - inter
            ovj = inter / denom if denom > 0 else 0.0
        else:
            ovr = ovj = 0.0

        tor = len(qs & ts) / qs_len if qs else 0.0
        dor = len(qs & ds) / qs_len if qs else 0.0
        por = len(qs & ps) / qs_len if qs else 0.0

        item_c3 = item_char3_sets[ip]
        if q_char3 and item_c3:
            inter_c3 = len(q_char3 & item_c3)
            denom_c3 = qc3_len + len(item_c3) - inter_c3
            c3 = inter_c3 / denom_c3 if denom_c3 > 0 else 0.0
        else:
            c3 = 0.0

        X[i, 0] = sf
        X[i, 1] = rf
        X[i, 2] = st
        X[i, 3] = rt
        X[i, 4] = sc3
        X[i, 5] = rc3
        X[i, 6] = ovr
        X[i, 7] = ovj
        X[i, 8] = tor
        X[i, 9] = dor
        X[i, 10] = por
        X[i, 11] = c3
        X[i, 12] = 1 if q_loc == str(item_loc_arr[ip]) else 0
        X[i, 13] = 1 if q_cat == str(item_microcat_arr[ip]) else 0
        X[i, 14] = X[i, 13]
        X[i, 15] = float(item_rating_arr[ip])
        X[i, 16] = float(item_reviews_arr[ip])
        X[i, 17] = float(item_price_log_arr[ip])
        X[i, 18] = float(item_title_len_arr[ip])
        X[i, 19] = float(item_desc_len_arr[ip])

    return item_pos_list, X

# Проверка, что порядок feature_cols совпадает с индексами выше:
print("Проверка порядка feature_cols:")
for i, c in enumerate(feature_cols):
    print(f"  {i:2d}: {c}")

In [ ]:
import random
random.seed(42)
N_EVAL = 1000
eval_q_positions = random.sample(range(len(valid_groups)), N_EVAL)

recalls_v2 = []
start = time.time()
for qp in tqdm(eval_q_positions, desc="Honest Recall@50 v2 (fast)"):
    item_pos_list, X_q = compute_features_for_query_fast(qp)
    scores = final_model.predict_proba(X_q)[:, 1]
    top_idx = np.argsort(-scores)[:50]
    top_ids = {item_ids_array[item_pos_list[i]] for i in top_idx}
    relevant = set(valid_groups.iloc[qp]['relevant_items'])
    recalls_v2.append(len(top_ids & relevant) / len(relevant) if relevant else 0.0)

recalls_v2 = np.array(recalls_v2)
print(f"\nЧестный Recall@50 v2 на {N_EVAL} запросах:")
print(f"  mean    = {recalls_v2.mean():.4f}")
print(f"  median  = {np.median(recalls_v2):.4f}")
print(f"  Recall=0: {(recalls_v2 == 0).mean():.4f}")
print(f"  Recall=1: {(recalls_v2 == 1).mean():.4f}")
print(f"  Время: {time.time() - start:.1f} сек.")
print(f"\nV1 было (на 5000): 0.7458") 

In [ ]:
### from 03_inference

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import time
import pickle
from tqdm import tqdm
import bm25s

# --- Загрузка модели v2 ---
with open('../dataset/lgbm_reranker_v2.pkl', 'rb') as f:
    payload = pickle.load(f)
final_model = payload['model']
feature_cols = payload['feature_cols']
print(f"Модель v2 загружена, {len(feature_cols)} признаков")

# --- BM25 индексы (full, title, char3) ---
r_full = bm25s.BM25.load('../dataset/bm25_index', load_corpus=False)
r_title = bm25s.BM25.load('../dataset/bm25_index_title', load_corpus=False)
r_char3 = bm25s.BM25.load('../dataset/bm25_index_char3', load_corpus=False)
print("BM25 индексы загружены")

# --- Токены полей для всех item'ов ---
tokens_by_id = {}
with open('../dataset/item_tokens_fields.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        tokens_by_id[obj['item_id']] = obj

# --- Char-3-gram токены для item'ов ---
item_char3_by_id = {}
with open('../dataset/item_title_char3.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        item_char3_by_id[obj['item_id']] = set(obj['tokens'])

print(f"Токены загружены для {len(tokens_by_id)} item'ов")

# --- Токенизация benchmark_queries ---
stemmer = RussianStemmer()
TOKEN_RE = re.compile(r'[а-яa-z0-9]+')
_STEM_CACHE = {}

def tokenize_stem(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) <= 1:
            continue
        s = _STEM_CACHE.get(t)
        if s is None:
            s = stemmer.stem(t)
            _STEM_CACHE[t] = s
        out.append(s)
    return out

def char_3gram_tokens(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9]', '', text)
    if len(text) < 3:
        return [text] if text else []
    return [text[i:i+3] for i in range(len(text) - 2)]

benchmark_queries['q_tokens'] = [tokenize_stem(t) for t in tqdm(
    benchmark_queries['search_query'].tolist(), desc="tokenize queries"
)]
benchmark_queries['q_char3'] = [char_3gram_tokens(t) for t in tqdm(
    benchmark_queries['search_query'].tolist(), desc="char3 queries"
)]

print("Пример:", benchmark_queries['search_query'].iloc[0])
print("Токены:", benchmark_queries['q_tokens'].iloc[0][:10])
print("Char-3:", benchmark_queries['q_char3'].iloc[0][:10])

In [ ]:
K_BM25 = 1000
BATCH = 128

def batch_bm25(retriever, q_tokens_list, k, name):
    all_ids = [None] * len(q_tokens_list)
    all_scores = [None] * len(q_tokens_list)
    for s in tqdm(range(0, len(q_tokens_list), BATCH), desc=f"BM25 {name}"):
        e = min(s + BATCH, len(q_tokens_list))
        batch_q = q_tokens_list[s:e]
        safe = [q if q else ['_empty_'] for q in batch_q]
        res, sc = retriever.retrieve(safe, k=k, show_progress=False)
        for j in range(len(batch_q)):
            all_ids[s + j] = res[j]
            all_scores[s + j] = sc[j]
    return all_ids, all_scores

q_tokens_list = benchmark_queries['q_tokens'].tolist()
q_char3_list = benchmark_queries['q_char3'].tolist()

print("BM25 full top-1000...")
bm25_full_ids, bm25_full_scores = batch_bm25(r_full, q_tokens_list, K_BM25, "full")
print("BM25 title top-1000...")
bm25_title_ids, bm25_title_scores = batch_bm25(r_title, q_tokens_list, K_BM25, "title")
print("BM25 char3 top-1000...")
bm25_char3_ids, bm25_char3_scores = batch_bm25(r_char3, q_char3_list, K_BM25, "char3")
print("BM25 поиск завершён")

In [ ]:
# Приводим числовые поля benchmark_items
def to_float(s):
    if pd.isna(s): return np.nan
    s = str(s).replace(',', '.').replace(' ', '')
    try: return float(s)
    except: return np.nan

benchmark_items['item_price'] = benchmark_items['item_price'].apply(to_float)
benchmark_items['item_rating'] = pd.to_numeric(benchmark_items['item_rating'], errors='coerce').fillna(-1)
benchmark_items['item_rating_reviews_count'] = pd.to_numeric(
    benchmark_items['item_rating_reviews_count'], errors='coerce'
).fillna(-1)

bi = benchmark_items.reset_index(drop=True)

item_ids_array = bi['item_id'].values
item_microcat_arr = bi['item_microcat_id'].values
item_loc_arr = bi['item_location_id'].values
item_rating_arr = bi['item_rating'].values
item_reviews_arr = bi['item_rating_reviews_count'].values
item_price_log_arr = np.log1p(bi['item_price'].fillna(0).clip(lower=0).values)
item_title_len_arr = bi['item_title_raw'].fillna('').astype(str).str.len().values
item_desc_len_arr = bi['item_description_raw'].fillna('').astype(str).str.len().values

# Токены полей в позиционных массивах
item_tokens_title_list = [tokens_by_id.get(iid, {}).get('title', []) for iid in item_ids_array]
item_tokens_desc_list = [tokens_by_id.get(iid, {}).get('desc', []) for iid in item_ids_array]
item_tokens_params_list = [tokens_by_id.get(iid, {}).get('params', []) for iid in item_ids_array]

# Char-3-gram токены по позициям
item_char3_sets = [item_char3_by_id.get(iid, set()) for iid in item_ids_array]

print("Lookup-массивы готовы")
print(f"item_ids_array: {len(item_ids_array)}")
print(f"item_char3_sets: {len(item_char3_sets)}")

In [ ]:
def score_rank_for(ids_arr, scores_arr, qp, ip):
    ids = ids_arr[qp]
    where = np.where(ids == ip)[0]
    if len(where) == 0:
        return 9999, 0.0
    rank = int(where[0])
    return rank, float(scores_arr[qp][rank])

q_loc_arr = benchmark_queries['search_location_id'].astype(str).tolist()
q_cat_arr = benchmark_queries['search_category'].astype(str).tolist()
q_char3_arr = benchmark_queries['q_char3'].tolist()

def compute_features_for_query(qp):
    """Считает фичи для всех кандидатов из union(full, title, char3) top-1000."""
    pool = set(bm25_full_ids[qp].tolist()) | \
           set(bm25_title_ids[qp].tolist()) | \
           set(bm25_char3_ids[qp].tolist())
    item_pos_list = list(pool)
    X = np.zeros((len(item_pos_list), len(feature_cols)), dtype=np.float32)

    qt = q_tokens_list[qp]
    qs = set(qt)
    q_char3 = set(q_char3_arr[qp])
    q_loc = q_loc_arr[qp]
    q_cat = q_cat_arr[qp]

    for i, ip in enumerate(item_pos_list):
        tt = item_tokens_title_list[ip]
        dt = item_tokens_desc_list[ip]
        pt = item_tokens_params_list[ip]
        at = set(tt + dt + pt)
        ts = set(tt); ds = set(dt); ps = set(pt)

        rf, sf = score_rank_for(bm25_full_ids, bm25_full_scores, qp, ip)
        rt, st = score_rank_for(bm25_title_ids, bm25_title_scores, qp, ip)
        rc3, sc3 = score_rank_for(bm25_char3_ids, bm25_char3_scores, qp, ip)

        if qs and at:
            inter = len(qs & at)
            ovr = inter / len(qs)
            ovj = inter / len(qs | at)
        else:
            ovr = ovj = 0.0

        tor = len(qs & ts) / len(qs) if qs else 0.0
        dor = len(qs & ds) / len(qs) if qs else 0.0
        por = len(qs & ps) / len(qs) if qs else 0.0

        item_char3 = item_char3_sets[ip]
        if q_char3 and item_char3:
            c3 = len(q_char3 & item_char3) / len(q_char3 | item_char3)
        else:
            c3 = 0.0

        X[i] = [
            sf, rf, st, rt, sc3, rc3,
            ovr, ovj, tor, dor, por, c3,
            int(q_loc == str(item_loc_arr[ip])),
            int(q_cat == str(item_microcat_arr[ip])),
            int(q_cat == str(item_microcat_arr[ip])),
            float(item_rating_arr[ip]),
            float(item_reviews_arr[ip]),
            float(item_price_log_arr[ip]),
            int(item_title_len_arr[ip]),
            int(item_desc_len_arr[ip]),
        ]
    return item_pos_list, X

In [ ]:
K_FINAL = 50
N_QUERIES = len(benchmark_queries)

predictions = []
start = time.time()

for qp in tqdm(range(N_QUERIES), desc="Rerank v2 benchmark_queries"):
    item_pos_list, X_q = compute_features_for_query(qp)
    scores = final_model.predict_proba(X_q)[:, 1]
    top_idx = np.argsort(-scores)[:K_FINAL]
    top_item_ids = [item_ids_array[item_pos_list[i]] for i in top_idx]
    predictions.append(top_item_ids)

elapsed = time.time() - start
print(f"Инференс v2 занял {elapsed:.1f} сек. ({elapsed/60:.1f} мин.)")
print(f"Средняя длина ответа: {np.mean([len(p) for p in predictions]):.1f}")

In [ ]:
query_ids = benchmark_queries['query_id'].tolist()
assert len(query_ids) == len(predictions), "Несовпадение числа запросов"

answer_df = pd.DataFrame({
    'query_id': query_ids,
    'answer': [' '.join(p[:K_FINAL]) for p in predictions],
})

print(f"Строк: {len(answer_df)}")
print(answer_df.head(3))

In [ ]:
assert (answer_df['query_id'].str.len() == 16).all(), "query_id должен быть 16 символов"
assert answer_df['query_id'].nunique() == len(answer_df), "Есть дубли query_id"

all_item_ids = set()
for ans in answer_df['answer']:
    ids = ans.split(' ')
    assert len(ids) <= 50, "Больше 50 item_id в строке"
    assert len(ids) == len(set(ids)), "Повторы внутри строки"
    for iid in ids:
        assert len(iid) == 16, f"item_id не 16 символов: {iid}"
        assert iid == iid.lower(), f"item_id не в нижнем регистре: {iid}"
        assert re.match(r'^[0-9a-f]{16}$', iid), f"item_id не hex: {iid}"
        all_item_ids.add(iid)

valid_ids = set(benchmark_items['item_id'])
missing = all_item_ids - valid_ids
assert len(missing) == 0, f"item_id не найдены в корпусе: {len(missing)}"

print("Все проверки пройдены")
print(f"Уникальных item_id в ответах: {len(all_item_ids)}")

In [ ]:
answer_df.to_csv('answer_v2.csv', index=False)
print("Сохранено в answer_v2.csv")

# Контрольная перечитка
check = pd.read_csv('../answer_v2.csv', dtype=str)
print("\nКолонки:", check.columns.tolist())
print("Строк:", len(check))
for _, row in check.head(2).iterrows():
    n_items = len(row['answer'].split(' '))
    print(f"  {row['query_id']} -> {n_items} item_id")